# Model Sonuçlarını Tek Tabloda Karşılaştırma

Bu notebook **Prophet, ARIMA, XGBoost ve LightGBM** sonuçlarını yeniden model kurmadan, diğer notebookların **kaydedilmiş çıktı tablolarından** çeker.

Kullanılan dosyalar:

- `2_prophet_baseline.ipynb`
- `3_arima_baseline.ipynb`
- `4_xgboost_forecasting.ipynb`

Bu üç notebookun önce çalıştırılmış ve **çıktılarıyla birlikte kaydedilmiş** olması gerekir.

Karşılaştırma, bütün modellerde kullanılan son 4 haftalık test dönemi için **MAE** ve **RMSE** üzerinden yapılır.


In [1]:
import json
from io import StringIO
from pathlib import Path

import pandas as pd


In [2]:
# Karşılaştırma notebook'u diğer üç notebook ile aynı klasörde bulunmalıdır.

BASE_DIR = Path.cwd()

NOTEBOOKS = {
    "Prophet": BASE_DIR / "2_prophet_baseline.ipynb",
    "ARIMA": BASE_DIR / "3_arima_baseline.ipynb",
    "XGBoost_LightGBM": BASE_DIR / "4_xgboost_forecasting.ipynb"
}

missing_files = [
    str(path.name)
    for path in NOTEBOOKS.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Şu notebook dosyaları bulunamadı: "
        + ", ".join(missing_files)
        + "\nKarşılaştırma notebook'unu diğer üç notebook ile aynı klasöre koy."
    )

NOTEBOOKS


{'Prophet': WindowsPath('c:/Users/sanug/OneDrive/Desktop/trend_forecasting/notebooks/2_prophet_baseline.ipynb'),
 'ARIMA': WindowsPath('c:/Users/sanug/OneDrive/Desktop/trend_forecasting/notebooks/3_arima_baseline.ipynb'),
 'XGBoost_LightGBM': WindowsPath('c:/Users/sanug/OneDrive/Desktop/trend_forecasting/notebooks/4_xgboost_forecasting.ipynb')}

## Diğer notebooklardan sonuç tablosunu çekme

Aşağıdaki fonksiyon `.ipynb` dosyasını JSON olarak açar, istediğimiz sonuç hücresini bulur ve hücrenin kaydedilmiş HTML tablo çıktısını `pandas.DataFrame` olarak okur.

Böylece model sonuçlarını elle kopyalayıp yapıştırmaya gerek kalmaz.


In [3]:
def extract_saved_dataframe(notebook_path, source_marker):
    """Notebook içinde belirli kodu içeren hücrenin kaydedilmiş tablo çıktısını okur."""

    with open(notebook_path, "r", encoding="utf-8") as file:
        notebook = json.load(file)

    matching_cells = []

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        if source_marker in source:
            matching_cells.append(cell)

    if not matching_cells:
        raise ValueError(
            f"{notebook_path.name} içinde '{source_marker}' bulunan hücre bulunamadı."
        )

    # Aynı ifade birden fazla hücrede varsa en sondakini kullan.
    for cell in reversed(matching_cells):
        outputs = cell.get("outputs", [])

        for output in reversed(outputs):
            data = output.get("data", {})

            if "text/html" in data:
                html = data["text/html"]

                if isinstance(html, list):
                    html = "".join(html)

                result = pd.read_html(StringIO(html))[0]

                # Jupyter'ın tablo indexinden oluşan gereksiz sütunu kaldır.
                result = result.loc[
                    :,
                    ~result.columns.astype(str).str.startswith("Unnamed")
                ]

                return result

    raise ValueError(
        f"{notebook_path.name} içindeki ilgili hücrenin kaydedilmiş tablo çıktısı yok. "
        "Önce o notebook'u baştan çalıştırıp kaydet."
    )


In [4]:
# Prophet sonuçlarını çekiyoruz.
prophet_metrics = extract_saved_dataframe(
    NOTEBOOKS["Prophet"],
    "metrics_df.round(2)"
)

prophet_metrics = prophet_metrics[
    ["keyword", "MAE", "RMSE"]
].copy()

prophet_metrics["model"] = "Prophet"

prophet_metrics


,keyword,MAE,RMSE,model
0,SQL,5.09,5.78,Prophet
1,Python,21.56,22.87,Prophet


In [5]:
# ARIMA sonuçlarını çekiyoruz.
arima_metrics = extract_saved_dataframe(
    NOTEBOOKS["ARIMA"],
    "metrics_df.round(2)"
)

arima_metrics = arima_metrics[
    ["keyword", "MAE", "RMSE"]
].copy()

arima_metrics["model"] = "ARIMA"

arima_metrics


,keyword,MAE,RMSE,model
0,SQL,4.39,4.65,ARIMA
1,Python,5.23,6.98,ARIMA


In [6]:
# XGBoost ve LightGBM sonuçları aynı karşılaştırma tablosunda bulunduğu için
# ikisini birlikte çekiyoruz.
ml_metrics = extract_saved_dataframe(
    NOTEBOOKS["XGBoost_LightGBM"],
    "model_comparison_df.round(2)"
)

ml_metrics = ml_metrics[
    ["keyword", "model", "MAE", "RMSE"]
].copy()

ml_metrics


,keyword,model,MAE,RMSE
0,Python,XGBoost,6.19,7.95
1,Python,LightGBM,12.05,13.91
2,SQL,LightGBM,3.23,3.37
3,SQL,XGBoost,3.50,3.78


## Tek Karşılaştırma Tablosu

Her anahtar kelime bir satırdır. Prophet, ARIMA, XGBoost ve LightGBM'in MAE/RMSE değerleri yan yana gösterilir.

- **MAE düşükse daha iyi**
- **RMSE düşükse daha iyi**
- `En İyi Model (MAE)` ilgili anahtar kelimede en düşük MAE değerine sahip modeli gösterir.


In [7]:
# Dört modelin sonucunu aynı uzun-form tabloda birleştiriyoruz.

all_metrics = pd.concat(
    [
        prophet_metrics[["keyword", "model", "MAE", "RMSE"]],
        arima_metrics[["keyword", "model", "MAE", "RMSE"]],
        ml_metrics[["keyword", "model", "MAE", "RMSE"]]
    ],
    ignore_index=True
)

all_metrics["MAE"] = pd.to_numeric(all_metrics["MAE"], errors="coerce")
all_metrics["RMSE"] = pd.to_numeric(all_metrics["RMSE"], errors="coerce")

all_metrics


,keyword,model,MAE,RMSE
0,SQL,Prophet,5.09,5.78
1,Python,Prophet,21.56,22.87
2,SQL,ARIMA,4.39,4.65
3,Python,ARIMA,5.23,6.98
4,Python,XGBoost,6.19,7.95
5,Python,LightGBM,12.05,13.91
6,SQL,LightGBM,3.23,3.37
7,SQL,XGBoost,3.50,3.78


In [8]:
# Sonuçları tek ve okunaklı bir wide tabloya dönüştürüyoruz.

model_order = ["Prophet", "ARIMA", "XGBoost", "LightGBM"]

comparison_table = (
    all_metrics
    .pivot(index="keyword", columns="model", values=["MAE", "RMSE"])
)

comparison_table.columns = [
    f"{model}_{metric}"
    for metric, model in comparison_table.columns
]

desired_columns = []

for model in model_order:
    desired_columns.extend([
        f"{model}_MAE",
        f"{model}_RMSE"
    ])

comparison_table = comparison_table.reindex(
    columns=desired_columns
)

# Her anahtar kelime için en iyi modeli MAE'ye göre buluyoruz.
best_mae_rows = (
    all_metrics
    .loc[all_metrics.groupby("keyword")["MAE"].idxmin()]
    .set_index("keyword")
)

comparison_table["En İyi Model (MAE)"] = best_mae_rows["model"]
comparison_table["En İyi MAE"] = best_mae_rows["MAE"]

# RMSE'ye göre en iyi modeli de gösteriyoruz.
best_rmse_rows = (
    all_metrics
    .loc[all_metrics.groupby("keyword")["RMSE"].idxmin()]
    .set_index("keyword")
)

comparison_table["En İyi Model (RMSE)"] = best_rmse_rows["model"]
comparison_table["En İyi RMSE"] = best_rmse_rows["RMSE"]

comparison_table = (
    comparison_table
    .reset_index()
    .round(2)
)

comparison_table


,keyword,Prophet_MAE,Prophet_RMSE,ARIMA_MAE,ARIMA_RMSE,XGBoost_MAE,XGBoost_RMSE,LightGBM_MAE,LightGBM_RMSE,En İyi Model (MAE),En İyi MAE,En İyi Model (RMSE),En İyi RMSE
0,Python,21.56,22.87,5.23,6.98,6.19,7.95,12.05,13.91,ARIMA,5.23,ARIMA,6.98
1,SQL,5.09,5.78,4.39,4.65,3.50,3.78,3.23,3.37,LightGBM,3.23,LightGBM,3.37


### Yorumlama

Bu tabloda bir modelin daha iyi olması için hata değerlerinin daha düşük olması gerekir. Anahtar kelimelerin ölçekleri birbirinden farklı olabileceği için modeli değerlendirirken yalnızca genel ortalamaya değil, **anahtar kelime bazındaki MAE ve RMSE değerlerine** bakmak daha sağlıklıdır.
